In [4]:
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')
HUGGING_FACE_TOKEN = userdata.get('HUGGING_FACE_TOKEN')

In [3]:
%pip install python-dotenv
%pip install OpenAI

In [5]:
!wget https://raw.githubusercontent.com/saintdragon2/gpt_agent_2025_easyspub/main/chap05/audio/lsy_audio_2023_58s.mp3

--2025-07-22 11:21:15--  https://raw.githubusercontent.com/saintdragon2/gpt_agent_2025_easyspub/main/chap05/audio/lsy_audio_2023_58s.mp3
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 928958 (907K) [audio/mpeg]
Saving to: ‘lsy_audio_2023_58s.mp3’

lsy_audio_2023_58s. 100%[===================>] 907.19K  --.-KB/s    in 0.02s   

2025-07-22 11:21:15 (39.6 MB/s) - ‘lsy_audio_2023_58s.mp3’ saved [928958/928958]



In [6]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

In [7]:
# mp3 음성을 텍스트로 변환


# mp3 파일 경로 설정
audio_file_path='/content/lsy_audio_2023_58s.mp3'

# mp3 음성 파일을 텍스트로 변환한 결과(파일을 읽어야 한다)
# 파일 읽을 때 -> r / 텍스트 -> t(생략가능) / 그림, 소리, 동영상 등 (텍스트 X) -> b
with open(audio_file_path, 'rb') as audio_file:
  transcription = client.audio.transcriptions.create(
      model='whisper-1',
      file=audio_file
  )

transcription

Transcription(text='안녕하세요. 이 강의는 GPT API로 챗봇 만들기라는 내용을 다루는 강의입니다. GPT API에 대해서 생소하신 분들도 있을 텐데 우리가 잘 알고 있는 채GPT, 채GPT 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 뭐 이런 강의들이 사실 많이 있습니다. 그래서 여러 가지들이 있는데 좀 이 강의의 특징이라고 한다면 GPT로 명확한 미션을 달성하는 챗봇 프로그램을 만드는 게 사실 쉽지는 않은데 이걸 어떻게 해서 구현을 하는지 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로 예제는 여러 가지가 될 수 있는데 여기서 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서 생성하는 프로그램 만드는 것을 다루려고 합니다. 그래서 프로그램이 실행되는 모습을 한번 보여드릴게요. 우리가 만들 프로그램은 이런 식으로 이제 나타나게 되고', logprobs=None, usage=UsageDuration(seconds=58.0, type='duration'))

In [8]:
# 한국어 음성을 영어로 번역하기
with open(audio_file_path, 'rb') as audio_file:
  translation = client.audio.translations.create(
      model='whisper-1',
      file=audio_file
  )
translation

Translation(text="Hello, this is a lecture on how to make a chatbot with GPT API. Some of you may be unfamiliar with GPT API. We're going to talk about how to make the program we want using the chat GPT function that we know well. There are a lot of lectures like this. There are a lot of things, but if I were to say the characteristics of this lecture, it's not easy to make a chatbot program that achieves a clear mission with GPT. I'm going to talk about how to implement this and why it's necessary. As an example, there are many examples. The example here is to create a program that automatically creates a music playlist video through conversation. I'll show you how the program runs. This is how the program we're going to make appears.")

In [9]:
!pip list

Package                               Version
------------------------------------- -------------------
absl-py                               1.4.0
accelerate                            1.9.0
aiofiles                              24.1.0
aiohappyeyeballs                      2.6.1
aiohttp                               3.11.15
aiosignal                             1.4.0
alabaster                             1.0.0
albucore                              0.0.24
albumentations                        2.0.8
ale-py                                0.11.2
altair                                5.5.0
annotated-types                       0.7.0
antlr4-python3-runtime                4.9.3
anyio                                 4.9.0
argon2-cffi                           25.1.0
argon2-cffi-bindings                  21.2.0
array_record                          0.7.2
arviz                                 0.21.0
astropy                               7.1.0
astropy-iers-data                     0.2025.7.14.0.

In [12]:
# 허깅페이스에서 제공하는 코드 복사 -> 일부 수정
# 허깅페이스 'whisper-large-v3' 검색
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer, # 토크나이저 : 텍스트를 작은 단위로 쪼개는 도구
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True,
    chunk_length_s=10,
    stride_length_s = 2,
)

sample = '/content/lsy_audio_2023_58s.mp3'
result = pipe(sample)
print(result)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
/usr/local/lib/python3.11/dist-packages/transformers/models/whisper/generation_whisper.py:604: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation

{'text': ' 안녕하세요. 이 강의는 GPT-API로 챗봇 만들기 라는 내용을 다루는 강의입니다. GPT-API에 대해서 생소하신 분들도 있을텐데 우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 이런 강의들이 사실 많이 있습니다. 그래서 여러가지들이 있는데 이 강의 특징이라고 한다면 GPT로 명확한 미션을 달성하는 챕터 프로그램을 만드는게 사실 쉽지는 않은데 이걸 어떻게 해서 구현을 하는지 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로 예제는 여러가지가 될 수 있는데 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서 생성하는 프로그램을 만드는 것을 다루려고 합니다. 프로그램이 실행되는 모습을 한번 보여드릴게요. 우리가 만들 프로그램은 이런 식으로 이제 나타나게 되고', 'chunks': [{'timestamp': (0.0, 6.3), 'text': ' 안녕하세요. 이 강의는 GPT-API로 챗봇 만들기 라는 내용을 다루는 강의입니다.'}, {'timestamp': (7.18, 10.0), 'text': ' GPT-API에 대해서 생소하신 분들도 있을텐데'}, {'timestamp': (11.0, 17.0), 'text': ' 우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서'}, {'timestamp': (17.0, 20.0), 'text': ' 우리가 원하는 프로그램을 어떻게 만드는지에 대해서'}, {'timestamp': (20.0, 22.0), 'text': ' 이야기할 거예요.'}, {'timestamp': (22.0, 24.0), 'text': ' 그래서 이런 강의들이 사실 많이 있습니다.'}, {'timestamp': (24.0, 27.48), 'text': ' 그래서 여러가지들이 있는데 이 강의 특징이라고 한다면'}, {'timestamp': (27.48, 29.58), 'text': ' G

In [13]:
# chunks(청크)를 csv파일로 저장하기
# 청크 : 덩어리, 컴퓨터 공학에서는 일정한 크기의 저장공간

# {'chunks':[{'timestamp'} : (0.0, 6.3), 'text':'']} 형태로 chunks부분 출력
start_end_text = []
for chunk in result['chunks']:
  start = chunk['timestamp'][0] # 시작 시각
  end = chunk['timestamp'][1] # 끝 시각
  text = chunk['text'] # 내용
  start_end_text.append([start, end, text])

import pandas as pd
df = pd.DataFrame(start_end_text, columns=['start', 'end', 'text']) # 데이터프레임으로 생성
df.to_csv('audio.csv', index=False, sep='|') # 데이터프레임을 csv 파일로 내보내기
display(df)

,start,end,text
0,0.00,6.30,안녕하세요. 이 강의는 GPT-API로 챗봇 만들기 라는 내용을 다루는 강의입니다.
1,7.18,10.00,GPT-API에 대해서 생소하신 분들도 있을텐데
2,11.00,17.00,"우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서"
3,17.00,20.00,우리가 원하는 프로그램을 어떻게 만드는지에 대해서
4,20.00,22.00,이야기할 거예요.
5,22.00,24.00,그래서 이런 강의들이 사실 많이 있습니다.
6,24.00,27.48,그래서 여러가지들이 있는데 이 강의 특징이라고 한다면
7,27.48,29.58,GPT로 명확한 미션을 달성하는
8,29.58,31.66,챕터 프로그램을 만드는게 사실
9,31.66,34.32,쉽지는 않은데 이걸 어떻게 해서


In [4]:
%pip install pyannote.audio
%pip install numpy==1.26

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 107.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.1 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.3.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.1 which is incompatible.
cupy-cuda12x 13.3.0 requires numpy<2.3,>=1.22, but you have numpy 2.3.1 which is incompatible.


  Using cached numpy-1.26.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (58 kB)
Using cached numpy-1.26.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.1
    Uninstalling numpy-2.3.1:
      Successfully uninstalled numpy-2.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.0 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.0 which is incompatible.


In [2]:
from pyannote.audio import Pipeline

# 파이프라인 --> 여러처리단계를 연결한 작업흐름
pipeline = Pipeline.from_pretrained(
  "pyannote/speaker-diarization-3.1",
  use_auth_token="HUGGING_FACE_TOKEN")

HfHubHTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/pyannote/speaker-diarization-3.1/resolve/main/config.yaml (Request ID: Root=1-687f86af-26f08c2e40d789ac41ef86cf;5c68630e-86b4-423d-a610-c9a77cd8532a)

Invalid credentials in Authorization header

In [1]:
%pip install onnxruntime-gpu

  Using cached numpy-2.3.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.2/283.2 MB 3.2 MB/s eta 0:00:00
Using cached numpy-2.3.1-cp311-cp311-manylinux_2_28_x86_64.whl (16.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.1 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.3.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.1 which is incompatible.
cupy-cuda12x 13.3.0 requires numpy<2.3,>=1.22, but you have numpy 2.3.1 which is incompatible.


In [3]:
import torch

# cuda가 사용 가능한 경우 cuda를 사용하도록 설정
if torch.cuda.is_available():
    pipeline.to(torch.device("cuda"))
    print('cuda is available')
else:
    print('cuda is not available')

NameError: name 'pipeline' is not defined

In [2]:
!wget https://raw.githubusercontent.com/saintdragon2/gpt_agent_2025_easyspub/main/chap05/audio/싼기타_비싼기타.mp3

--2025-07-22 12:41:48--  https://raw.githubusercontent.com/saintdragon2/gpt_agent_2025_easyspub/main/chap05/audio/%EC%8B%BC%EA%B8%B0%ED%83%80_%EB%B9%84%EC%8B%BC%EA%B8%B0%ED%83%80.mp3
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6968467 (6.6M) [audio/mpeg]
Saving to: ‘싼기타_비싼기타.mp3’

싼기타_비싼기타.mp3 100%[===================>]   6.65M  --.-KB/s    in 0.06s   

2025-07-22 12:41:48 (117 MB/s) - ‘싼기타_비싼기타.mp3’ saved [6968467/6968467]



In [ ]:
# run the pipeline on an audio file
diarization = pipeline("/content/싼기타_비싼기타.mp3")

# dump the diarization output to disk using RTTM format
with open("/content/싼기타_비싼기타.rttm", "w") as rttm:
    diarization.write_rttm(rttm)